In [1]:
import feedparser 
url = "https://www.cert.ssi.gouv.fr/feed/" 
rss_feed = feedparser.parse(url) 
entries_link = []
for entry in rss_feed.entries: 
    print("Titre :", entry.title) 
    print("Description:", entry.description) 
    print("Lien :", entry.link) 
    entries_link.append(entry.link)
    print("Date :", entry.published)

Titre : Multiples vulnérabilités dans Microsoft Azure Linux (05 juin 2026)
Description: De multiples vulnérabilités ont été découvertes dans Microsoft Azure Linux. Elles permettent à un attaquant de provoquer un problème de sécurité non spécifié par l'éditeur.
Lien : https://www.cert.ssi.gouv.fr/avis/CERTFR-2026-AVI-0693/
Date : Fri, 05 Jun 2026 00:00:00 +0000
Titre : Vulnérabilité dans Cisco Catalyst SD-WAN (05 juin 2026)
Description: Une vulnérabilité a été découverte dans Cisco Catalyst SD-WAN. Elle permet à un attaquant de provoquer une élévation de privilèges. Cisco indique que la vulnérabilité CVE-2026-20245 est activement exploitée.
Lien : https://www.cert.ssi.gouv.fr/avis/CERTFR-2026-AVI-0699/
Date : Fri, 05 Jun 2026 00:00:00 +0000
Titre : Multiples vulnérabilités dans le noyau Linux de SUSE (05 juin 2026)
Description: De multiples vulnérabilités ont été découvertes dans le noyau Linux de SUSE. Certaines d'entre elles permettent à un attaquant de provoquer une atteinte à la con

In [2]:
import requests 
import re 
cve_ids = []
for link in entries_link:
    url = link + "json/"
    
    try:
        response = requests.get(url)
        # Lève une erreur si le code HTTP n'est pas 200 (ex: 404, 500)
        response.raise_for_status() 
        
        # Tente de décoder le JSON
        data = response.json() 
        
    except requests.exceptions.RequestException as e:
        # Attrape les erreurs de connexion, timeout, 404, etc.
        print(f"Erreur de connexion ou HTTP pour {url} : {e}")
        continue # Passe au lien suivant
        
    except requests.exceptions.JSONDecodeError:
        # Attrape le problème que tu avais (réponse non-JSON)
        print(f"Réponse JSON invalide pour {url}. Le site a renvoyé du texte brut ou du HTML.")
        # Optionnel : décommenter la ligne suivante pour inspecter le problème en direct
        # print("Contenu reçu :", response.text[:200]) 
        continue # Passe au lien suivant

    # --- Tout ce qui est en dessous ne s'exécute que si le try a réussi ---
    
    # Extraction des CVE références dans la clé cves du dict data 
    # (Sécurisé avec .get() au cas où la clé "cves" serait absente)
    cves_data = data.get("cves", [])
    print("CVE référencés :", cves_data) 

    
    #Si on a des CVE, on extrait proprement les identifiants 'name'
    if len(cves_data) > 0:
        for cve in cves_data:
            if "name" in cve.keys():
                cve_ids.append(cve["name"])
    else:
        print("Aucun CVE référencé trouvé pour ce lien.")
    
    #Extraction des CVE avec une regex (ton code actuel)
    cve_pattern = r"CVE-\d{4}-\d{4,7}" 
    cve_list = list(set(re.findall(cve_pattern, str(data)))) 
    print("CVE trouvés :", cve_list)
    print("-" * 40)

CVE référencés : [{'name': 'CVE-2026-42250', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-42250'}, {'name': 'CVE-2026-9149', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-9149'}, {'name': 'CVE-2026-9150', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-9150'}]
CVE trouvés : ['CVE-2026-9150', 'CVE-2026-42250', 'CVE-2026-9149']
----------------------------------------
CVE référencés : [{'name': 'CVE-2026-20245', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-20245'}]
CVE trouvés : ['CVE-2026-20245']
----------------------------------------
CVE référencés : [{'name': 'CVE-2026-43366', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-43366'}, {'name': 'CVE-2026-23260', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-23260'}, {'name': 'CVE-2026-23447', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-23447'}, {'name': 'CVE-2026-23387', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-23387'}, {'name': 'CVE-2026-31658', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-316

In [3]:

from get_cve_score import get_cve_scores
for cve_id in cve_ids:
    print(f"Traitement de {cve_id}...")
    cve_info = get_cve_scores(cve_id)
    
    print(f"=== Informations pour le {cve_info['identifiant']} ===")
    print(f"Gravité           : {cve_info['base_severity']}")
    print(f"Score CVSS         : {cve_info['score_cvss']}")
    print(f"Score EPSS         : {cve_info['score_epss']}")
    print(f"Code CWE           : {cve_info['type_cwe']}")
    print(f"Produits affectés  : {cve_info['produits_affectes']}")
    print(f"Description        : {cve_info['description']}")
    print("-" * 50)

Traitement de CVE-2026-42250...
    -> [API MITRE] Interrogation pour CVE-2026-42250...
    -> [API FIRST] Interrogation EPSS pour CVE-2026-42250...
=== Informations pour le CVE-2026-42250 ===
Gravité           : Non renseigné
Score CVSS         : None
Score EPSS         : 0.000210000
Code CWE           : CWE-787
Produits affectés  : [{'vendor': 'bzip2', 'produit': 'bzip2', 'versions': '0'}]
Description        : bzip2 contains an off‑by‑one error in the bzip2recover utility. When processing a specially crafted file, the application performs an out‑of‑bounds write to a global buffer, resulting in memory corruption and a crash (denial of service).

This issue was fixed in bzip2 patch 35d122a3df8b0cc4082a4d89fdc6ee99f375fe67
--------------------------------------------------
Traitement de CVE-2026-9149...
    -> [API MITRE] Interrogation pour CVE-2026-9149...
    -> [API FIRST] Interrogation EPSS pour CVE-2026-9149...
=== Informations pour le CVE-2026-9149 ===
Gravité           : Non rens

KeyboardInterrupt: 